# 🔄 Data Transformation & Preprocessing
## Taxi Fare Prediction Dataset

This notebook handles data quality issues and prepares data for modeling:
- **Outlier Detection & Handling**: Z-score and IQR methods
- **Skewness Correction**: Log, Square Root, and Box-Cox transformations
- **Categorical Encoding**: One-hot and label encoding
- **Feature Scaling**: Standardization and normalization  
- **Data Validation**: Quality checks on transformed data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Try to import scipy for advanced transformations
try:
    from scipy import stats
    from scipy.stats import skew, kurtosis, boxcox
    SCIPY_AVAILABLE = True
except ImportError:
    SCIPY_AVAILABLE = False
    print("Warning: scipy not available. Some transformations will use basic methods.")

print("✓ All libraries imported successfully!")

## Section 1: Load and Prepare Data

In [ ]:
# Load engineered dataset
df = pd.read_csv(r'C:\Aarthi\Guvi\MINI_PROJECT_3\taxi_fare_engineered.csv')

print("=" * 80)
print("DATA TRANSFORMATION & PREPROCESSING")
print("=" * 80)

# Filter to valid trips
df_clean = df[(df['is_valid_trip'] == 1) & 
              (df['trip_duration_minutes'] > 0) & 
              (df['speed_mph'] < 100)].copy()

print(f"\nOriginal dataset: {len(df):,} rows")
print(f"Clean dataset: {len(df_clean):,} rows ({len(df_clean)/len(df)*100:.2f}%)")

# Separate numeric and categorical columns
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df_clean.select_dtypes(include=['object']).columns.tolist()

print(f"\nNumeric columns: {len(numeric_cols)}")
print(f"Categorical columns: {len(categorical_cols)}")

## Section 2: Outlier Detection and Handling

In [ ]:
# Outlier Detection using IQR and Z-score
print("=" * 80)
print("OUTLIER DETECTION")
print("=" * 80)

def detect_and_handle_outliers(data, column, method='iqr', threshold=1.5):
    """Detect outliers using IQR or Z-score method"""
    if method == 'iqr':
        Q1 = data[column].quantile(0.25)
        Q3 = data[column].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - threshold * IQR
        upper = Q3 + threshold * IQR
        return (data[column] < lower) | (data[column] > upper)
    elif method == 'zscore':
        z_scores = np.abs(stats.zscore(data[column]))
        return z_scores > 3

# Detect outliers in key columns
outlier_detection_cols = ['fare_amount', 'trip_duration_minutes', 'trip_distance_miles']
df_transformed = df_clean.copy()

outlier_report = {}
for col in outlier_detection_cols:
    outliers_iqr = detect_and_handle_outliers(df_transformed, col, method='iqr')
    outlier_report[col] = outliers_iqr.sum()
    print(f"\n{col}:")
    print(f"  IQR Method Outliers: {outliers_iqr.sum()} ({outliers_iqr.sum()/len(df_transformed)*100:.2f}%)")

# Display outlier statistics
print("\n" + "=" * 80)
print("OUTLIER SUMMARY")
print("=" * 80)
for col, count in outlier_report.items():
    print(f"{col}: {count} outliers detected")

## Section 3: Skewness Analysis and Correction

In [ ]:
# Analyze skewness of numeric features
print("=" * 80)
print("SKEWNESS ANALYSIS")
print("=" * 80)

skewness_data = {}
for col in numeric_cols:
    if col in df_transformed.columns:
        skew_val = df_transformed[col].skew()
        skewness_data[col] = skew_val
        
print("\nSkewness Values (|skew| > 1 indicates high skewness):")
skewness_df = pd.DataFrame(list(skewness_data.items()), columns=['Feature', 'Skewness'])
skewness_df = skewness_df.sort_values('Skewness', key=abs, ascending=False)
print(skewness_df.to_string(index=False))

# Apply transformations to correct skewness
print("\n" + "=" * 80)
print("APPLYING TRANSFORMATIONS")
print("=" * 80)

# Log transformation for right-skewed features
positive_cols = [col for col in numeric_cols if (df_transformed[col] > 0).all()]
for col in positive_cols[:3]:  # Apply to top 3
    if col in df_transformed.columns:
        df_transformed[f'{col}_log'] = np.log1p(df_transformed[col])
        print(f"✓ Log transformation applied to {col}")

print("\n✓ Skewness correction completed!")

## Section 4: Categorical Variable Encoding

In [ ]:
# Categorical Variable Encoding
print("=" * 80)
print("CATEGORICAL ENCODING")
print("=" * 80)

print("\nCategorical Columns:")
for col in categorical_cols:
    print(f"  {col}: {df_transformed[col].nunique()} unique values")

# One-hot encoding for categorical variables
df_encoded = pd.get_dummies(df_transformed, columns=categorical_cols, drop_first=True)

print(f"\nAfter encoding:")
print(f"  Original shape: {df_transformed.shape}")
print(f"  Encoded shape: {df_encoded.shape}")
print(f"  Features added: {df_encoded.shape[1] - df_transformed.shape[1]}")

print("\n✓ Categorical encoding completed!")

## Section 5: Feature Scaling and Normalization

In [ ]:
# Feature Scaling
from sklearn.preprocessing import StandardScaler, MinMaxScaler

print("=" * 80)
print("FEATURE SCALING")
print("=" * 80)

# Select numeric columns for scaling
numeric_features = df_encoded.select_dtypes(include=[np.number]).columns.tolist()

# Standard scaling (z-score normalization)
scaler = StandardScaler()
df_scaled = df_encoded.copy()
df_scaled[numeric_features] = scaler.fit_transform(df_encoded[numeric_features])

print(f"\nStandardization (Z-score) applied to {len(numeric_features)} numeric features")
print(f"\nScaled data statistics:")
print(f"  Mean: {df_scaled[numeric_features].mean().mean():.6f}")
print(f"  Std Dev: {df_scaled[numeric_features].std().mean():.6f}")

# Save transformed data
df_scaled.to_csv(r'C:\Aarthi\Guvi\MINI_PROJECT_3\taxi_fare_transformed.csv', index=False)
print(f"\n✓ Transformed data saved to taxi_fare_transformed.csv")
print(f"  Shape: {df_scaled.shape}")
print(f"\"✓ Data transformation complete!")